# Gate 2: train the learned P-frame coder on Vimeo-90K

`docs/PLAN_learned_pframe.md`, gate 2. Gate 1 (the laptop pilot on DAVIS TRAIN) passed on
2026-11-05: the learned P-frame coder cost **0.43x** an I-frame at matched PSNR, **-37.2%**
BD-rate against today's hand-built P-frame path (`outputs/p1_pframe/README.md`). That model
had only seen DAVIS. This notebook trains it on Vimeo-90K septuplets, so DAVIS VAL-B becomes
genuinely out-of-domain, and then trains it on **chains of its own reconstructions** so errors
stop piling up along a GOP.

**This notebook is a thin wrapper.** All the logic is `scripts/p1_pframe.py train-vimeo`,
which is unit-tested (`tests/test_p1_pframe.py`). The notebook mounts Drive, finds Kaggle
credentials, installs the package and runs that script.

## What it runs

1. **Stage 1, one P-step.** I-frame = the frozen Phase B intra codec; one P-frame coded from
   its warped reconstruction. Each lambda's P-coder starts as a copy of its own intra codec
   (training from scratch collapses - see the pilot README).
2. **Stage 2, unrolled.** Starts from stage 1. Each clip is an I-frame plus `UNROLL`
   P-frames, each referencing the previous *reconstruction*, so the coder learns from (and
   pays for) its own accumulated error.

One P-coder per intra lambda (1e-3, 3e-4, 1e-4). Each Vimeo chunk is downloaded once and all
three lambdas train on it before it is deleted.

## Before you start: put the three intra checkpoints on Drive

Each P-coder is paired with the Phase B intra checkpoint for its lambda (86 MB each). On your
laptop they are in `outputs/stage1_vimeo/colab_run5_phase_b/`. Upload that **whole folder**
into `My Drive/neural_streaming_colab/` so Drive has

    neural_streaming_colab/colab_run5_phase_b/phase_b_unitstep_lambda1e-3/checkpoints/best.pt
    neural_streaming_colab/colab_run5_phase_b/phase_b_unitstep_lambda3e-4/checkpoints/best.pt
    neural_streaming_colab/colab_run5_phase_b/phase_b_unitstep_lambda1e-4/checkpoints/best.pt

(If the original Phase B run's folder `stage1_vimeo_64ch_qat/` is already on this Drive, the
config cell finds the checkpoints there instead.) Also needed: `kaggle.json` (cached on Drive
after the first upload), and a GPU runtime.

## If the session disconnects

Reopen and **Run All**. `progress.json` records which (lambda, chunk) pairs are finished and
they are skipped; an unfinished one resumes from its `latest.pt` (saved every
`SAVE_EVERY` steps, optimizer included). Everything lives on Drive.

## What this does *not* do

Measure the gate. VAL-B is DAVIS, which is on your laptop, not here. The last cell zips the
trained models; on the laptop, `p1_pframe.py gate --models-dir ...` scores them (section 7).

## 0. Configuration

In [ ]:
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_DIR = globals().get('WORK_DIR') or (Path('/content') if IN_COLAB else Path.cwd().resolve())
REPO_URL = 'https://github.com/yuvidewan/neural_streaming.git'
REPO_DIR = WORK_DIR / 'neural_streaming'
SCRATCH_DIR = WORK_DIR / 'vimeo_scratch'      # the dataset, one chunk at a time, deleted as it goes

if IN_COLAB:
    DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/neural_streaming_colab')
else:
    DRIVE_PROJECT_DIR = WORK_DIR
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'pframe_gate2'   # a fresh folder: earlier runs own theirs
STAGE1_DIR = OUTPUT_DIR / 'stage1'
STAGE2_DIR = OUTPUT_DIR / 'stage2'
ZIP_PATH = WORK_DIR / 'pframe_gate2_models.zip'

# The Phase B intra checkpoints, one per lambda ({} is the lambda tag). The first pattern
# whose three files all exist wins.
INTRA_CANDIDATES = [
    str(DRIVE_PROJECT_DIR / 'colab_run5_phase_b' / 'phase_b_unitstep_lambda{}' / 'checkpoints' / 'best.pt'),
    str(DRIVE_PROJECT_DIR / 'stage1_vimeo_64ch_qat' / 'phase_b_unitstep_lambda{}' / 'checkpoints' / 'best.pt'),
]

LAMBDAS = ['1e-3', '3e-4', '1e-4']   # the rate points; the gate needs all three

# --- Stage 1: one P-step per clip ---
# The pilot trained 20k steps on DAVIS TRAIN. Two chunks at 10k each is the same number of
# steps on ~6x more (and different) footage; the second chunk runs at lr/10.
STAGE1_CHUNKS = [1, 2]
STAGE1_STEPS_PER_CHUNK = 10000

# --- Stage 2: chains of UNROLL P-frames on the coder's own reconstructions ---
# A GOP is 10 frames; 3 chained P-frames is where the pilot README expects error growth to
# show, at ~3x the cost of a stage-1 step. One fresh chunk, at lr/10 (the last chunk decays).
RUN_STAGE2 = True
STAGE2_CHUNKS = [3]
STAGE2_STEPS_PER_CHUNK = 4000
UNROLL = 3

BATCH = 8            # the pilot's batch
CROP = 256           # Vimeo frames are 448x256
LEARNING_RATE = 1e-4
SAVE_EVERY = 500     # steps between latest.pt saves (Drive): the most a disconnect can lose
SEED = 42
NUM_WORKERS = 2

KAGGLE_DATASET_OWNER = 'wangsally'
KAGGLE_DATASET_PREFIX = 'vimeo-90k'

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f'Output folder: {OUTPUT_DIR}')
print(f'Stage 1: chunks {STAGE1_CHUNKS} x {STAGE1_STEPS_PER_CHUNK} steps, lambdas {LAMBDAS}')
print(f'Stage 2: {"chunks %s x %d steps, unroll %d" % (STAGE2_CHUNKS, STAGE2_STEPS_PER_CHUNK, UNROLL) if RUN_STAGE2 else "off"}')

## 1. Mount Drive and find the intra checkpoints

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INTRA_CHECKPOINT = None
for pattern in INTRA_CANDIDATES:
    if all(Path(pattern.format(lam)).is_file() for lam in LAMBDAS):
        INTRA_CHECKPOINT = pattern
        break
if INTRA_CHECKPOINT is None:
    missing = '\n'.join(f'  {pattern.format(lam)}' for pattern in INTRA_CANDIDATES for lam in LAMBDAS)
    raise FileNotFoundError(
        'The Phase B intra checkpoints are not on Drive. Upload the laptop folder\n'
        'outputs/stage1_vimeo/colab_run5_phase_b into My Drive/neural_streaming_colab/\n'
        f'and run this cell again. Looked for:\n{missing}')
print(f'Intra checkpoints: {INTRA_CHECKPOINT}')
for lam in LAMBDAS:
    path = Path(INTRA_CHECKPOINT.format(lam))
    print(f'  {lam}: {path.stat().st_size / 2**20:.0f} MB')

for stage in (STAGE1_DIR, STAGE2_DIR):
    progress = stage / 'progress.json'
    if progress.is_file():
        print(f'NOTE: {progress} exists - finished (lambda, chunk) pairs will be skipped. '
              'Right after a disconnect; wrong if this folder is from a different run.')

## 2. Kaggle API credentials

In [ ]:
import shutil

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json_target = kaggle_dir / 'kaggle.json'
next_to_notebook = WORK_DIR / 'kaggle.json'

if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
    print('Using KAGGLE_USERNAME / KAGGLE_KEY from the environment.')
elif kaggle_json_target.is_file():
    print(f'Using the existing Kaggle token at {kaggle_json_target}')
elif next_to_notebook.is_file():
    shutil.copy(next_to_notebook, kaggle_json_target)
    print(f'Copied {next_to_notebook} -> {kaggle_json_target}')
elif IN_COLAB:
    drive_kaggle_json = DRIVE_PROJECT_DIR / 'kaggle.json'
    if drive_kaggle_json.is_file():
        shutil.copy(drive_kaggle_json, kaggle_json_target)
        print(f'Reused Kaggle API token from {drive_kaggle_json}')
    else:
        print('No saved Kaggle token found on Drive - upload your kaggle.json now.')
        print('(Get it from https://www.kaggle.com/settings -> API -> Create New Token)')
        from google.colab import files
        uploaded = files.upload()
        shutil.move(next(iter(uploaded)), kaggle_json_target)
        shutil.copy(kaggle_json_target, drive_kaggle_json)
        print(f'Saved a copy to {drive_kaggle_json} - future reconnects reuse it.')
else:
    raise RuntimeError('No Kaggle credentials found: put kaggle.json next to this notebook, '
                       'or set KAGGLE_USERNAME and KAGGLE_KEY.')
if kaggle_json_target.is_file():
    os.chmod(kaggle_json_target, 0o600)

## 3. Get the code and install it

In [ ]:
import subprocess
import sys

if REPO_DIR.is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))
print(f'cwd: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q kaggle

### OpenCV on a headless server

Same fix as the Stage 1 notebook: if `import cv2` fails for missing system libraries, swap
every OpenCV build for the headless one. Changes nothing where OpenCV already works.

In [ ]:
from importlib import metadata


def cv2_imports():
    probe = subprocess.run([sys.executable, '-c', 'import cv2; print(cv2.__version__)'],
                           capture_output=True, text=True)
    detail = probe.stdout.strip() if probe.returncode == 0 else probe.stderr.strip().splitlines()[-1]
    return probe.returncode == 0, detail


def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


ok, detail = cv2_imports()
if ok:
    print(f'cv2 {detail} imports fine - nothing to do.')
else:
    version = installed_version('opencv-python') or installed_version('opencv-python-headless')
    spec = f'opencv-python-headless=={version}' if version else 'opencv-python-headless>=4.9,<5.0'
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'opencv-python',
                    'opencv-python-headless', 'opencv-contrib-python',
                    'opencv-contrib-python-headless'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec], check=True)
    ok, detail = cv2_imports()
    if not ok:
        raise RuntimeError(f'cv2 still does not import: {detail}')
    print(f'cv2 {detail} imports fine now.')

### Sanity check and time estimate

Raises on a CPU runtime. Times real stage-1 and stage-2 training steps (motion search
included) on random data and prints how long the whole run will take on this GPU. For
reference, an RTX 5060 laptop measured **0.72 s** per stage-1 step and **2.1 s** per stage-2
step at unroll 3.

In [ ]:
import importlib.util
import time

import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA device: Runtime -> Change runtime type -> GPU.')
print(f'GPU: {torch.cuda.get_device_name(0)}')

spec = importlib.util.spec_from_file_location('p1_pframe', REPO_DIR / 'scripts' / 'p1_pframe.py')
p1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(p1)
from nvc.training.checkpoint import load_model_from_checkpoint

device = torch.device('cuda')
intra, _ = load_model_from_checkpoint(Path(INTRA_CHECKPOINT.format(LAMBDAS[0])), device=device)
intra.eval().requires_grad_(False)
model = p1.ConditionalPFrameCoder.from_intra(intra).to(device).train()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def seconds_per_step(unroll, repeats=5):
    clip = torch.rand(BATCH, unroll + 1, 3, CROP, CROP, device=device)
    for i in range(repeats + 2):          # two warm-up steps
        if i == 2:
            torch.cuda.synchronize()
            started = time.time()
        optimizer.zero_grad(set_to_none=True)
        p1.unrolled_step(model, intra, clip, 3e-4)
        optimizer.step()
    torch.cuda.synchronize()
    return (time.time() - started) / repeats


s1 = seconds_per_step(1)
s2 = seconds_per_step(UNROLL) if RUN_STAGE2 else 0.0
print(f'Peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB')
print(f'stage 1: {s1:.2f} s/step   stage 2 (unroll {UNROLL}): {s2:.2f} s/step')
hours1 = s1 * STAGE1_STEPS_PER_CHUNK * len(STAGE1_CHUNKS) * len(LAMBDAS) / 3600
hours2 = (s2 * STAGE2_STEPS_PER_CHUNK * len(STAGE2_CHUNKS) * len(LAMBDAS) / 3600) if RUN_STAGE2 else 0.0
downloads = len(STAGE1_CHUNKS) + (len(STAGE2_CHUNKS) if RUN_STAGE2 else 0)
print(f'Estimated training time: stage 1 {hours1:.1f} h + stage 2 {hours2:.1f} h '
      f'= {hours1 + hours2:.1f} h, plus {downloads} chunk downloads (~10-20 min each).')
del model, optimizer, intra
torch.cuda.empty_cache()

## 4. Stage 1: one P-step per clip

Re-running this cell after a disconnect resumes; finished (lambda, chunk) pairs are skipped.
Each lambda writes `stage1/lambda<L>/model.pt` (what the gate reads), `latest.pt` (to
resume) and `history.json`.

In [ ]:
COMMON = [
    sys.executable, '-u', 'scripts/p1_pframe.py', 'train-vimeo',
    '--scratch-dir', str(SCRATCH_DIR),
    '--lambdas', *LAMBDAS,
    '--intra-checkpoint', INTRA_CHECKPOINT,
    '--batch', str(BATCH),
    '--crop', str(CROP),
    '--lr', str(LEARNING_RATE),
    '--lr-decay-chunks', '1',
    '--save-every', str(SAVE_EVERY),
    '--seed', str(SEED),
    '--num-workers', str(NUM_WORKERS),
    '--kaggle-owner', KAGGLE_DATASET_OWNER,
    '--kaggle-prefix', KAGGLE_DATASET_PREFIX,
]
subprocess.run(COMMON + ['--output-dir', str(STAGE1_DIR),
                         '--chunks', *[str(c) for c in STAGE1_CHUNKS],
                         '--steps-per-chunk', str(STAGE1_STEPS_PER_CHUNK),
                         '--unroll', '1'], check=True)

## 5. Stage 2: chains of P-frames on the coder's own reconstructions

Starts each lambda from its stage-1 `model.pt` (`--init-from`) and writes to `stage2/`, so
stage 1's models stay untouched and both can be gated.

In [ ]:
if not RUN_STAGE2:
    print('RUN_STAGE2 is False - skipping.')
else:
    missing = [lam for lam in LAMBDAS if not (STAGE1_DIR / f'lambda{lam}' / 'model.pt').is_file()]
    if missing:
        raise FileNotFoundError(f'Stage 1 has not finished for lambdas {missing}: run section 4 first.')
    subprocess.run(COMMON + ['--output-dir', str(STAGE2_DIR),
                             '--chunks', *[str(c) for c in STAGE2_CHUNKS],
                             '--steps-per-chunk', str(STAGE2_STEPS_PER_CHUNK),
                             '--unroll', str(UNROLL),
                             '--init-from', str(STAGE1_DIR)], check=True)

### Where training got to

In [ ]:
import json

for stage in (STAGE1_DIR, STAGE2_DIR):
    progress = stage / 'progress.json'
    print(f'{stage.name}: ' + (str(json.loads(progress.read_text())['done']) if progress.is_file()
                              else 'nothing finished yet'))
    for lam in LAMBDAS:
        history = stage / f'lambda{lam}' / 'history.json'
        if history.is_file():
            last = json.loads(history.read_text())[-1]
            print(f"  {lam}: chunk {last['chunk']} step {last['step']}  {last['psnr']:.2f} dB  "
                  f"{last['bpp']:.4f} bpp (training proxy, not the gate)")

## 6. Zip the trained models

Only `model.pt`, `history.json` and `progress.json` from both stages (about 35 MB per model);
`latest.pt` stays on Drive. Then download the zip.

In [ ]:
import zipfile

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for stage in (STAGE1_DIR, STAGE2_DIR):
        for path in sorted(stage.rglob('*')):
            if path.name in ('model.pt', 'history.json', 'progress.json'):
                bundle.write(path, Path('pframe_gate2') / path.relative_to(OUTPUT_DIR))
with zipfile.ZipFile(ZIP_PATH) as bundle:
    assert bundle.testzip() is None, 'the archive is corrupt'
    for entry in bundle.infolist():
        print(f'  {entry.file_size / 2**20:8.1f} MB  {entry.filename}')
print(ZIP_PATH)
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))

## 7. Then, on the laptop: the gate

Unzip into the repo so the models land in `outputs/p1_pframe/colab/pframe_gate2/`, then score
each stage on DAVIS VAL-B against the same intra and hand-built rows as gate 1 (the VAL-B
caches from gate 1 are reused):

```bash
python scripts/p1_pframe.py gate --models-dir outputs/p1_pframe/colab/pframe_gate2/stage1 --output-name gate2_stage1.json
python scripts/p1_pframe.py gate --models-dir outputs/p1_pframe/colab/pframe_gate2/stage2 --output-name gate2_stage2.json
```

Gate 2 passes if the learned P-frame still costs <= 0.45x an I-frame at matched PSNR, now
that VAL-B is out of the training domain. The pilot's DAVIS-trained number to compare with is
**0.427x** (-37.2% BD-rate vs the hand-built path).